<a href="https://colab.research.google.com/github/Bee-Ene/shri-qlearning-execute-or-clarify/blob/main/UserS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

USER STUDY ANALYSIS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import os, glob, warnings
warnings.filterwarnings('ignore')

#PATHS
LOG_DIR      = "/PATH/session_logs"
GS_CSV       = "/PATH/GodspeedTest.csv"
OUT          = "/PATH/user_study_results"
QTABLE_PATH  = "/PATH/q_table.npy"
os.makedirs(OUT, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams.update({
    'font.family':    'DejaVu Sans',
    'figure.dpi':     150,
    'savefig.dpi':    300,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
})
BLUE   = "#2563eb"; RED    = "#dc2626"; GREEN  = "#16a34a"
ORANGE = "#d97706"; PURPLE = "#7c3aed"; GRAY   = "#6b7280"


# LOAD SESSION LOGS

def load_sessions():
    files = sorted(glob.glob(os.path.join(LOG_DIR, "session_*.csv")))
    if not files:
        raise
        FileNotFoundError(
            f"No session_*.csv files found in {LOG_DIR}"
        )
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f)
            dfs.append(df)
        except Exception as e:
            print(f"  Warning: could not read {f}: {e}")
    df = pd.concat(dfs, ignore_index=True)
    print(f"Loaded {len(files)} session files → {len(df)} episodes "
          f"from {df.user_id.nunique()} users.")

    # Ensure required columns exist with fallbacks
    if "n_clarifications" not in df.columns and "n_clar" in df.columns:
        df["n_clarifications"] = df["n_clar"]
    if "episode_duration_sec" not in df.columns:
        df["episode_duration_sec"] = 5.0  # fallback
    if "intent_parsed" not in df.columns:
        df["intent_parsed"] = "UNKNOWN"
    if "robotics_experience" not in df.columns:
        df["robotics_experience"] = "unknown"

    # Normalise outcome column
    if "episode_outcome" in df.columns:
        df["episode_outcome"] = df["episode_outcome"].fillna("UNKNOWN")
    else:
        df["episode_outcome"] = "UNKNOWN"

    return df


# LOAD AND PARSE GODSPEED CSV

def load_godspeed():
    """
    Loads GodSpeed CSV and maps columns by POSITION.

    Standard GodSpeed item order in this form:
      Impression items 0-4   → Anthropomorphism (5 items)
      Impression items 5-9   → Likeability (5 items)
      Impression items 10-14 → Perceived Intelligence (5 items)
      Emotional state 0-2    → Perceived Safety (3 items)

    """
    if not os.path.exists(GS_CSV):
        raise FileNotFoundError(f"GodSpeed CSV not found: {GS_CSV}")

    gs = pd.read_csv(GS_CSV)
    print(f"\nGodSpeed CSV loaded: {len(gs)} responses, {len(gs.columns)} columns")
    print("Columns:", gs.columns.tolist())

    # Separate impression and emotional state columns
    impression_cols = [c for c in gs.columns
                       if "impression" in c.lower() or
                       ("rate your impression" in c.lower())]
    emotional_cols  = [c for c in gs.columns
                       if "emotional" in c.lower() or
                       ("emotional state" in c.lower())]

    if not impression_cols:
        # Fallback: use all numeric-looking columns after Timestamp
        all_cols    = [c for c in gs.columns if c != "Timestamp"]
        # Remove any user-id looking column
        uid_like    = [c for c in all_cols
                       if any(w in c.lower() for w in ["user","id","name","email"])]
        numeric_cols= [c for c in all_cols if c not in uid_like]
        # Try to parse as numeric
        numeric_cols= [c for c in numeric_cols
                       if gs[c].dtype in [np.float64, np.int64] or
                       pd.to_numeric(gs[c], errors="coerce").notna().sum() > len(gs)*0.5]
        impression_cols = numeric_cols[:15] if len(numeric_cols) >= 15 else numeric_cols
        emotional_cols  = numeric_cols[15:18] if len(numeric_cols) >= 18 else []
        print(f"\nFallback column detection: "
              f"{len(impression_cols)} impression, {len(emotional_cols)} emotional")

    print(f"\nDetected {len(impression_cols)} impression items: "
          f"{impression_cols[:3]}...")
    print(f"Detected {len(emotional_cols)} emotional state items: "
          f"{emotional_cols}")

    def to_num(series):
        return pd.to_numeric(series, errors="coerce")

    # Map subscales by position
    if len(impression_cols) >= 15:
        gs["Anthropomorphism"]      = gs[impression_cols[0:5]].apply(to_num).mean(axis=1)
        gs["Likeability"]           = gs[impression_cols[5:10]].apply(to_num).mean(axis=1)
        gs["PerceivedIntelligence"] = gs[impression_cols[10:15]].apply(to_num).mean(axis=1)
    elif len(impression_cols) >= 5:
        # Partial mapping
        gs["Anthropomorphism"]      = gs[impression_cols[0:min(5,len(impression_cols))]].apply(to_num).mean(axis=1)
        if len(impression_cols) >= 10:
            gs["Likeability"]           = gs[impression_cols[5:10]].apply(to_num).mean(axis=1)
        if len(impression_cols) >= 15:
            gs["PerceivedIntelligence"]       = gs[impression_cols[10:15]].apply(to_num).mean(axis=1)

    if len(emotional_cols) >= 3:
        gs["PerceivedSafety"]       = gs[emotional_cols[0:3]].apply(to_num).mean(axis=1)

    # User ID column
    uid_col = None
    for c in gs.columns:
        if any(w in c.lower() for w in ["user_id","user id","userid","participant"]):
            uid_col = c; break
    if uid_col:
        gs["user_id"] = gs[uid_col].astype(str).str.strip()
        print(f"  User ID column: '{uid_col}'")
    else:
        # Assign sequential IDs matching session log format
        gs["user_id"] = [f"U{i+1:02d}" for i in range(len(gs))]
        print("  No user_id column found, assigning U01, U02... sequentially.")

    subscales = [c for c in ["Anthropomorphism","Likeability",
                              "PerceivedIntelligence","PerceivedSafety"]
                 if c in gs.columns]
    print(f"\nGodSpeed subscales successfully computed: {subscales}")
    for s in subscales:
        vals = gs[s].dropna()
        print(f"  {s}: mean={vals.mean():.2f}  std={vals.std():.2f}  "
              f"min={vals.min():.1f}  max={vals.max():.1f}  n={len(vals)}")
    return gs


# PER-USER METRICS

def per_user_metrics(df):
    rows = []
    for uid, g in df.groupby("user_id"):
        n       = len(g)
        ok      = (g.episode_outcome == "SUCCESS").sum()
        avg_cl  = g.n_clarifications.mean()
        avg_dur = g.episode_duration_sec.mean()
        fe_dur  = g[g.n_clarifications == 0].episode_duration_sec.mean()
        n_comp  = (g.n_clarifications == 0).sum()
        n_inc   = (g.n_clarifications > 0).sum()
        rows.append({
            "user_id":               uid,
            "age":                   g.age.iloc[0] if "age" in g.columns else "N/A",
            "gender":                g.gender.iloc[0] if "gender" in g.columns else "N/A",
            "robotics_experience":   g.robotics_experience.iloc[0]
                                     if "robotics_experience" in g.columns else "N/A",
            "n_episodes":            n,
            "success_rate":          round(ok/n, 3) if n > 0 else 0,
            "failure_rate":          round(1 - ok/n, 3) if n > 0 else 0,
            "avg_clarifications":    round(float(avg_cl), 2),
            "avg_duration_sec":      round(float(avg_dur), 2),
            "avg_direct_exec_sec":   round(float(fe_dur), 2) if not np.isnan(fe_dur) else 0,
            "complete_commands":     int(n_comp),
            "incomplete_commands":   int(n_inc),
        })
    m = pd.DataFrame(rows)
    m.to_csv(f"{OUT}/per_user_metrics.csv", index=False)
    print(f"\nPer-user metrics computed for {len(m)} users")
    print(m[["user_id","n_episodes","success_rate",
             "avg_clarifications"]].to_string(index=False))
    return m


# A. PERFORMANCE MEASURES

def plot_A1_success_rate(m):
    fig, ax = plt.subplots(figsize=(max(12, len(m)*0.7), 5))
    x    = np.arange(len(m))
    bars = ax.bar(x, m.success_rate*100, color=BLUE, alpha=0.85,
                  edgecolor="white")
    mean_s = m.success_rate.mean()*100
    ax.axhline(mean_s, color=RED, linestyle="--", linewidth=2,
               label=f"Mean: {mean_s:.1f}%")
    ax.set_xticks(x); ax.set_xticklabels(m.user_id, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("Task Success Rate (%)"); ax.set_ylim(0, 115)
    ax.set_title(f"Task Success Rate per User  (N={len(m)} participants)\n"
                 f"Overall mean: {mean_s:.1f}%  |  "
                 f"Min: {m.success_rate.min()*100:.1f}%  "
                 f"Max: {m.success_rate.max()*100:.1f}%")
    for b, v in zip(bars, m.success_rate*100):
        ax.text(b.get_x()+b.get_width()/2, v+1.5,
                f"{v:.0f}%", ha="center", fontsize=8, fontweight="bold")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUT}/A1_success_rate_per_user.png", bbox_inches="tight")
    plt.close(); print("A1 saved")


def plot_A2_clarification_rate(m):
    fig, ax = plt.subplots(figsize=(max(12, len(m)*0.7), 5))
    x    = np.arange(len(m))
    bars = ax.bar(x, m.avg_clarifications, color=ORANGE, alpha=0.85,
                  edgecolor="white")
    mean_c = m.avg_clarifications.mean()
    ax.axhline(mean_c, color=RED, linestyle="--", linewidth=2,
               label=f"Mean: {mean_c:.2f}")
    ax.set_xticks(x); ax.set_xticklabels(m.user_id, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("Avg. Clarification Requests per Episode")
    ax.set_title(f"Clarification Rate per User  (N={len(m)})\n"
                 f"Mean: {mean_c:.2f}  Lower = robot understood commands better")
    for b, v in zip(bars, m.avg_clarifications):
        ax.text(b.get_x()+b.get_width()/2, v+0.02,
                f"{v:.2f}", ha="center", fontsize=8)
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUT}/A2_clarification_rate_per_user.png", bbox_inches="tight")
    plt.close(); print("A2 saved")


def plot_A3_outcome_distribution(df):
    counts = df.episode_outcome.value_counts()
    label_map = {
        "SUCCESS":                 "Success",
        "FAILED_EXECUTION":        "Failed Execution",
        "FAILED_EXCESSIVE_CLARIF": "Failed (Max Clarif.)",
        "UNKNOWN":                 "Unknown/Other",
    }
    labels = [label_map.get(k, k) for k in counts.index]
    colors = [GREEN if "SUCCESS" in k else RED if "EXECUTION" in k
              else ORANGE if "CLARIF" in k else GRAY
              for k in counts.index]
    fig, ax = plt.subplots(figsize=(7, 7))
    wedges, texts, autotexts = ax.pie(
        counts.values, labels=labels, autopct="%1.1f%%",
        colors=colors, startangle=90,
        textprops={"fontsize": 11},
        wedgeprops={"edgecolor": "white", "linewidth": 1.5})
    for at in autotexts: at.set_fontweight("bold")
    total = counts.sum()
    ax.set_title(f"Episode Outcome Distribution\n"
                 f"All {len(df.user_id.unique())} users combined  "
                 f"({total} total episodes)")
    plt.tight_layout()
    plt.savefig(f"{OUT}/A3_outcome_distribution.png", bbox_inches="tight")
    plt.close(); print("A3 saved")


def plot_A4_combined_metrics(m):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    x = np.arange(len(m))
    w = 0.6

    for ax, col, lbl, clr, ymax in [
        (axes[0], "success_rate",       "Success Rate (%)",          BLUE,   115),
        (axes[1], "avg_clarifications", "Avg. Clarifications/ep",    ORANGE, None),
        (axes[2], "avg_duration_sec",   "Avg. Episode Duration (s)", PURPLE, None),
    ]:
        vals = m[col]*100 if col == "success_rate" else m[col]
        bars = ax.bar(x, vals, w, color=clr, alpha=0.85)
        ax.axhline(vals.mean(), color=RED, linestyle="--", linewidth=1.5, alpha=0.7)
        ax.set_xticks(x)
        ax.set_xticklabels(m.user_id, rotation=45, ha="right", fontsize=7)
        ax.set_title(lbl, fontsize=11, fontweight="bold")
        if ymax: ax.set_ylim(0, ymax)
        ax.grid(axis="y", alpha=0.3)

    plt.suptitle(f"Combined Performance Overview: {len(m)} Users",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUT}/A4_combined_performance.png", bbox_inches="tight")
    plt.close(); print("A4 saved")


# TIME AND STEPS

def plot_B1_episode_duration(df):
    fig, ax = plt.subplots(figsize=(14, 6))
    users   = sorted(df.user_id.unique())
    palette = plt.cm.tab20(np.linspace(0, 1, min(len(users), 20)))
    for i, uid in enumerate(users[:20]):  # cap at 20 for readability
        g = df[df.user_id == uid].sort_values("episode_number")
        ax.plot(g.episode_number, g.episode_duration_sec,
                marker="o", markersize=3, linewidth=1.2,
                color=palette[i], label=uid, alpha=0.75)
    ax.set_xlabel("Episode Number")
    ax.set_ylabel("Episode Duration (seconds)")
    ax.set_title(f"Episode Duration Over the Session  (N={len(users)} users)\n"
                 "Decreasing trend = user learned to give clearer commands")
    ax.legend(ncol=4, fontsize=7, loc="upper right"); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUT}/B1_episode_duration_over_session.png", bbox_inches="tight")
    plt.close(); print("B1 saved")


def plot_B2_time_metrics(m):
    fig, ax = plt.subplots(figsize=(max(12, len(m)*0.7), 5))
    x = np.arange(len(m)); w = 0.35
    b1 = ax.bar(x-w/2, m.avg_duration_sec, w,
                label="Avg Episode Duration", color=BLUE, alpha=0.85)
    b2 = ax.bar(x+w/2, m.avg_direct_exec_sec, w,
                label="Avg Direct Execution Duration", color=GREEN, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(m.user_id, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Duration (seconds)")
    ax.set_title("Time Metrics per User\n"
                 "Shorter direct execution = robot understood command immediately")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUT}/B2_time_metrics_per_user.png", bbox_inches="tight")
    plt.close(); print("B2 saved")


def plot_B3_clarification_steps(df):
    fig, ax = plt.subplots(figsize=(9, 5))
    max_c = int(df.n_clarifications.max()) if df.n_clarifications.max() > 0 else 3
    bins  = np.arange(-0.5, max_c+1.5)
    ax.hist(df.n_clarifications, bins=bins, color=PURPLE, alpha=0.8,
            edgecolor="white", linewidth=0.8)
    mean_c = df.n_clarifications.mean()
    ax.axvline(mean_c, color=RED, linestyle="--", linewidth=2,
               label=f"Mean: {mean_c:.2f}")
    ax.set_xlabel("Clarification Requests per Episode")
    ax.set_ylabel("Number of Episodes")
    ax.set_title(f"Clarification Steps Distribution\n"
                 f"0 = executed immediately  |  Mean: {mean_c:.2f}  "
                 f"|  Total episodes: {len(df)}")
    ax.set_xticks(range(max_c+1))
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUT}/B3_clarification_steps_distribution.png", bbox_inches="tight")
    plt.close(); print("B3 saved")


# C. COMMANDS PER USER

def plot_C1_command_completeness(m):
    fig, ax = plt.subplots(figsize=(max(12, len(m)*0.7), 5))
    x = np.arange(len(m)); w = 0.35
    ax.bar(x-w/2, m.complete_commands, w,
           label="Complete (no clarif. needed)", color=GREEN, alpha=0.85)
    ax.bar(x+w/2, m.incomplete_commands, w,
           label="Incomplete (clarif. triggered)", color=ORANGE, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(m.user_id, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Number of Commands")
    ax.set_title("Complete vs Incomplete Commands per User\n"
                 "Reflects natural variation in how users phrased commands")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUT}/C1_command_completeness_per_user.png", bbox_inches="tight")
    plt.close(); print("C1 saved")

'''
def plot_C2_intent_distribution(df):
    intents = df.intent_parsed.value_counts()
    fig, ax = plt.subplots(figsize=(10, 5))
    palette = [BLUE, GREEN, ORANGE, PURPLE, RED, GRAY, "#0891b2", "#be185d"]
    bars = ax.bar(range(len(intents)), intents.values,
                  color=palette[:len(intents)], alpha=0.85, edgecolor="white")
    ax.set_xticks(range(len(intents)))
    ax.set_xticklabels(intents.index, rotation=20, ha="right", fontsize=10)
    ax.set_ylabel("Number of Episodes")
    ax.set_title("Command Intent Distribution (All Users Combined)\n"
                 "Shows which command types were most commonly used")
    for b, v in zip(bars, intents.values):
        ax.text(b.get_x()+b.get_width()/2, v+0.3,
                str(v), ha="center", fontsize=9, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUT}/C2_intent_distribution.png", bbox_inches="tight")
    plt.close(); print("C2 saved")
'''

def plot_C3_most_clarified_commands(df):
    clar = df[df.n_clarifications > 0]
    if len(clar) == 0:
        print("C3: No clarification episodes found, skipping."); return
    top  = clar.command_text.value_counts().head(12)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(range(len(top)), top.values, color=RED, alpha=0.8, edgecolor="white")
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top.index, fontsize=9)
    ax.set_xlabel("Number of Episodes Requiring Clarification")
    ax.set_title("Commands Most Frequently Requiring Clarification\n"
                 "Identifies common ambiguity patterns in user commands")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUT}/C3_most_clarified_commands.png", bbox_inches="tight")
    plt.close(); print("C3 saved")


def plot_C4_command_heatmap(df):
    if df.intent_parsed.nunique() < 2:
        print("C4: Not enough intent variety, skipping."); return
    pivot = (df.groupby(["user_id","intent_parsed"])
               .size().unstack(fill_value=0))
    fig, ax = plt.subplots(figsize=(12, max(6, len(pivot)*0.4+2)))
    sns.heatmap(pivot, annot=True, fmt="d", cmap="Blues",
                linewidths=0.5, linecolor="white", ax=ax,
                cbar_kws={"label":"Number of Episodes"})
    ax.set_title("Command Intent Heatmap (Episodes per User per Intent)")
    ax.set_xlabel("Intent"); ax.set_ylabel("User ID")
    plt.tight_layout()
    plt.savefig(f"{OUT}/C4_command_intent_heatmap.png", bbox_inches="tight")
    plt.close(); print("C4 saved")


# D. DEMOGRAPHICS

def plot_D_demographics(df):
    demo = df.groupby("user_id").agg({
        "age":                   "first",
        "gender":                "first",
        "robotics_experience":   "first",
        "episode_number":        "max",
    }).reset_index()
    demo.columns = ["User ID","Age","Gender","Robotics Exp.","Episodes"]
    demo.to_csv(f"{OUT}/D_demographics.csv", index=False)

    fig, ax = plt.subplots(figsize=(13, max(4, len(demo)*0.45+1.5)))
    ax.axis("off")
    tbl = ax.table(
        cellText=demo.values, colLabels=demo.columns,
        cellLoc="center", loc="center",
        colColours=["#2E4057"]*len(demo.columns))
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1.1, 1.7)
    for (r,c), cell in tbl._cells.items():
        if r == 0:
            cell.set_text_props(color="white", fontweight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#f2f2f2")
    ax.set_title(f"Participant Demographics  (N={len(demo)})",
                 fontsize=13, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.savefig(f"{OUT}/D_demographics_table.png", bbox_inches="tight")
    plt.close(); print("D saved")
    return demo


# E. STATISTICAL CORRELATION

def plot_E1_correlation_matrix(m, gs):
    gs_cols   = [c for c in ["Anthropomorphism","Likeability",
                              "PerceivedIntelligence","PerceivedSafety"]
                 if c in gs.columns]
    perf_cols = ["success_rate","avg_clarifications","avg_duration_sec","failure_rate"]
    perf_lbl  = ["Task Success\nRate","Avg. Clarif.\n/Episode",
                 "Avg. Duration\n(s)","Failure\nRate"]

    if not gs_cols:
        print("E1: No GodSpeed subscales found, skipping correlation."); return

    merged = m.merge(gs[["user_id"]+gs_cols], on="user_id", how="inner")
    print(f"\nCorrelation analysis: {len(merged)} matched users "
          f"(session logs ∩ GodSpeed responses)")
    if len(merged) < 3:
        print("  Too few matched users for correlation, skipping E1."); return

    corr = pd.DataFrame(index=perf_lbl, columns=gs_cols, dtype=float)
    pval = pd.DataFrame(index=perf_lbl, columns=gs_cols, dtype=float)
    for pi, p in enumerate(perf_cols):
        for g_col in gs_cols:
            x = merged[p].dropna(); y = merged[g_col].dropna()
            idx = x.index.intersection(y.index)
            if len(idx) >= 3 and x[idx].std() > 0 and y[idx].std() > 0:
                r, pv = stats.spearmanr(x[idx], y[idx])
                corr.iloc[pi][g_col] = r
                pval.iloc[pi][g_col] = pv
            else:
                corr.iloc[pi][g_col] = np.nan
                pval.iloc[pi][g_col] = np.nan

    print("\nSpearman Correlation Matrix (r values):")
    print(corr.round(3).to_string())
    print("\nP-values:")
    print(pval.round(4).to_string())

    fig, ax = plt.subplots(figsize=(11, 6))
    mask = corr.isnull()
    sns.heatmap(corr.astype(float), annot=True, fmt=".2f",
               cmap="RdBu_r", center=0, vmin=-1, vmax=1,
               cbar_kws={"label":"Spearman r"},
               linewidths=0.5, linecolor="white",
               mask=mask, ax=ax)

    # Add significance stars
    for i in range(len(perf_cols)):
        for j, g_col in enumerate(gs_cols):
            p_val = pval.iloc[i][g_col]
            r_val = corr.iloc[i][g_col]
            if not np.isnan(p_val) and p_val < 0.05:
                ax.text(j+0.85, i+0.2, "*",
                        ha="center", va="center",
                        fontsize=14, color="black", fontweight="bold")

    ax.set_title(f"Spearman Correlation: Performance vs GodSpeed\n"
                 f"N={len(merged)} users  |  * = p<0.05  |  "
                 "Positive r = higher performance → higher perception score",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUT}/E1_correlation_matrix.png", bbox_inches="tight")
    plt.close()
    corr.to_csv(f"{OUT}/E1_r_values.csv")
    pval.to_csv(f"{OUT}/E1_p_values.csv")
    print("E1 saved")


def plot_E2_scatter(m, gs):
    gs_cols = [c for c in ["PerceivedIntelligence","Likeability","PerceivedSafety"]
               if c in gs.columns]
    if not gs_cols:
        print("E2: No GodSpeed subscales — skipping."); return
    merged = m.merge(gs[["user_id"]+gs_cols], on="user_id", how="inner")
    if len(merged) < 3:
        print("E2: Too few matched users."); return

    pairs = []
    if "PerceivedIntelligence" in gs_cols:
        pairs.append(("success_rate","PerceivedIntelligence",
                      "Task Success Rate (%)","Perceived Intelligence (1-5)"))
    if "Likeability" in gs_cols:
        pairs.append(("avg_clarifications","Likeability",
                      "Avg. Clarif. Requests/Episode","Likeability (1-5)"))
    if not pairs: return

    fig, axes = plt.subplots(1, len(pairs), figsize=(7*len(pairs), 5))
    if len(pairs) == 1: axes = [axes]

    for ax, (xc, yc, xl, yl) in zip(axes, pairs):
        xv = merged[xc]*100 if xc == "success_rate" else merged[xc]
        yv = merged[yc]
        ax.scatter(xv, yv, s=80, color=BLUE, alpha=0.8,
                  edgecolors="white", linewidth=0.5)
        for _, row in merged.iterrows():
            ax.annotate(row.user_id,
                       (row[xc]*100 if xc=="success_rate" else row[xc], row[yc]),
                       textcoords="offset points", xytext=(5,4), fontsize=7)
        if xv.std() > 0 and yv.std() > 0:
            r, p = stats.spearmanr(xv, yv)
            ax.set_title(f"{xl} vs {yl}\nSpearman r={r:.2f}  p={p:.3f}")
        else:
            ax.set_title(f"{xl} vs {yl}")
        ax.set_xlabel(xl); ax.set_ylabel(yl)
        ax.set_ylim(0.5, 5.5); ax.grid(alpha=0.3)

    plt.suptitle("Correlation Scatter Plots: Performance vs Perception",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUT}/E2_correlation_scatter.png", bbox_inches="tight")
    plt.close(); print("E2 saved")


# F. Q-LEARNING VISUALIZATION

def plot_F1_policy_plain_language():
    if not os.path.exists(QTABLE_PATH):
        print(f"F1: Q-table not found at {QTABLE_PATH} , skipping."); return
    Q = np.load(QTABLE_PATH)
    COMP_N = ["COMPLETE","PARTIAL","INCOMPLETE"]
    CLAR_N = ["CLEAR","AMBIGUOUS"]
    FEAS_N = ["FEASIBLE","UNCERTAIN","BLOCKED"]
    labels  = []
    for s in range(18):
        c=s//6; cl=(s%6)//3; f=s%3
        labels.append(f"[{s}] {COMP_N[c]} + {CLAR_N[cl]} + {FEAS_N[f]}")
    actions = np.argmax(Q, axis=1)
    colors  = [GREEN if a==0 else RED for a in actions]
    texts   = ["✓ ACT NOW" if a==0 else "? ASK FIRST" for a in actions]

    fig, ax = plt.subplots(figsize=(13, 10))
    ax.barh(range(18), [1]*18, color=colors, alpha=0.88,
           edgecolor="white", linewidth=1.0, height=0.82)
    ax.set_yticks(range(18)); ax.set_yticklabels(labels, fontsize=9)
    ax.set_xticks([]); ax.invert_yaxis()
    for i, (txt, col) in enumerate(zip(texts, colors)):
        ax.text(0.5, i, txt, ha="center", va="center",
                fontsize=10, fontweight="bold", color="white")
    ep = mpatches.Patch(color=GREEN, label="Robot acts immediately (EXECUTE)")
    cp = mpatches.Patch(color=RED,   label="Robot asks a question (CLARIFY)")
    ax.legend(handles=[ep,cp], loc="lower right", fontsize=11)
    ax.set_title("What the Robot Learned (Q-Learning Policy)\n"
                 "Only 1 of 18 states → Execute immediately. All others → Ask first.",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUT}/F1_policy_plain_language.png", bbox_inches="tight")
    plt.close(); print("F1 saved")


def plot_F2_qvalue_analysis():
    if not os.path.exists(QTABLE_PATH):
        print(f"F2: Q-table not found, skipping."); return
    Q       = np.load(QTABLE_PATH)
    actions = np.argmax(Q, axis=1)
    margin  = Q[:,0] - Q[:,1]
    COMP_N  = ["COMPL","PART","INCOMPL"]
    CLAR_N  = ["CLR","AMBIG"]
    FEAS_N  = ["FEAS","UNC","BLK"]
    short   = [f"[{s}]{COMP_N[s//6]}+{CLAR_N[(s%6)//3]}+{FEAS_N[s%3]}"
               for s in range(18)]

    fig, (ax1,ax2) = plt.subplots(1, 2, figsize=(16, 9))
    cols = [GREEN if m>0 else RED for m in margin]
    ax1.barh(range(18), margin, color=cols, alpha=0.85,
            edgecolor="white", height=0.7)
    ax1.axvline(0, color="black", linewidth=1.0)
    ax1.set_yticks(range(18)); ax1.set_yticklabels(short, fontsize=8)
    ax1.set_xlabel("Q(EXECUTE) − Q(CLARIFY)")
    ax1.set_title("Decision Margin per State\n(Positive = Execute preferred)")
    ax1.invert_yaxis(); ax1.grid(axis="x", alpha=0.3)

    reshape_m = margin.reshape(3,6)
    reshape_d = actions.reshape(3,6)
    annot = [[f"{'E' if reshape_d[r,c]==0 else 'C'}\n{reshape_m[r,c]:.1f}"
              for c in range(6)] for r in range(3)]
    xl    = [f"{CLAR_N[c//3]}+{FEAS_N[c%3]}" for c in range(6)]
    sns.heatmap(reshape_m, annot=annot, fmt="s",
               cmap="RdYlGn", ax=ax2, linewidths=0.5,
               xticklabels=xl,
               yticklabels=["COMPLETE","PARTIAL","INCOMPLETE"],
               cbar_kws={"label":"|Q_exec − Q_clar|"})
    ax2.set_title("Decision Confidence Heatmap\n"
                  "E=Execute, C=Clarify")
    plt.suptitle("Q-Learning Agent: Learned Q-values and Decision Confidence",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUT}/F2_qvalue_analysis.png", bbox_inches="tight")
    plt.close(); print("F2 saved")


# G. GODSPEED

def plot_G1_godspeed(gs):
    subscales = [c for c in ["Anthropomorphism","Likeability",
                              "PerceivedIntelligence","PerceivedSafety"]
                 if c in gs.columns]
    if not subscales:
        print("G1: No subscales found — skipping."); return
    means = gs[subscales].mean()
    stds  = gs[subscales].std()
    short = {"Anthropomorphism":"Anthropo-\nmorphism",
             "Likeability":"Likeability",
             "PerceivedIntelligence":"Perceived\nIntelligence",
             "PerceivedSafety":"Perceived\nSafety"}
    xlbls = [short.get(s,s) for s in subscales]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    bars = ax1.bar(xlbls, means, yerr=stds, capsize=5,
                  color=PURPLE, alpha=0.82, ecolor=GRAY,
                  error_kw={"linewidth":1.5})
    ax1.axhline(3.0, color=GRAY, linestyle="--", linewidth=1,
               label="Neutral midpoint (3.0)")
    ax1.set_ylim(0, 5.8); ax1.set_ylabel("Mean Score (1-5)")
    ax1.set_title("GodSpeed Subscale Scores\n(Mean ± Std Dev)")
    for b, m_val, s_val in zip(bars, means, stds):
        ax1.text(b.get_x()+b.get_width()/2, m_val+s_val+0.1,
                f"{m_val:.2f}", ha="center", fontsize=10, fontweight="bold")
    ax1.legend(); ax1.grid(axis="y", alpha=0.3)

    n      = len(subscales)
    angles = np.linspace(0, 2*np.pi, n, endpoint=False).tolist()
    vals   = means.tolist() + [means.tolist()[0]]
    angs   = angles + [angles[0]]
    ax2r   = plt.subplot(122, polar=True)
    ax2r.plot(angs, vals, color=PURPLE, linewidth=2.5)
    ax2r.fill(angs, vals, color=PURPLE, alpha=0.15)
    ax2r.set_xticks(angles)
    ax2r.set_xticklabels(xlbls, fontsize=9)
    ax2r.set_ylim(1, 5); ax2r.set_yticks([1,2,3,4,5])

    plt.suptitle(f"GodSpeed Results  (N={len(gs.dropna(subset=subscales[:1]))})",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUT}/G_godspeed_results.png", bbox_inches="tight")
    plt.close(); print("G1 saved")


def plot_G2_per_user(gs):
    subscales = [c for c in ["Anthropomorphism","Likeability",
                              "PerceivedIntelligence","PerceivedSafety"]
                 if c in gs.columns]
    if not subscales or "user_id" not in gs.columns: return
    data = gs.set_index("user_id")[subscales].dropna(how="all")
    if len(data) == 0: return
    fig, ax = plt.subplots(figsize=(12, max(5, len(data)*0.5+2)))
    sns.heatmap(data.astype(float), annot=True, fmt=".2f",
               cmap="YlOrRd", vmin=1, vmax=5,
               linewidths=0.5, linecolor="white",
               cbar_kws={"label":"Score (1-5)"}, ax=ax)
    ax.set_title("GodSpeed Scores per Participant\n"
                 "Higher score = more positive perception of robot")
    plt.tight_layout()
    plt.savefig(f"{OUT}/G2_godspeed_per_user.png", bbox_inches="tight")
    plt.close(); print("G2 saved")


# DISCUSSION

def write_user_study_discussion(m, gs, df):
    subscales = [c for c in ["Anthropomorphism","Likeability",
                              "PerceivedIntelligence","PerceivedSafety"]
                 if c in gs.columns]
    lines = []
    lines.append("="*65)
    lines.append("USER STUDY STATISTICAL ANALYSIS")
    lines.append("="*65)

    lines.append(f"\nSAMPLE: N={m.user_id.nunique()} participants, "
                 f"{len(df)} total episodes")

    lines.append("\n── PERFORMANCE AVERAGES ──")
    lines.append(f"  Mean task success rate:     {m.success_rate.mean()*100:.1f}% "
                 f"(std={m.success_rate.std()*100:.1f}%)")
    lines.append(f"  Min / Max success rate:     "
                 f"{m.success_rate.min()*100:.1f}% / {m.success_rate.max()*100:.1f}%")
    lines.append(f"  Mean clarif./episode:       {m.avg_clarifications.mean():.2f} "
                 f"(std={m.avg_clarifications.std():.2f})")
    lines.append(f"  Mean episode duration:      {m.avg_duration_sec.mean():.1f}s "
                 f"(std={m.avg_duration_sec.std():.1f}s)")
    lines.append(f"  Mean direct exec. duration: {m.avg_direct_exec_sec.mean():.1f}s")
    lines.append(f"  Complete commands:          "
                 f"{m.complete_commands.sum()} "
                 f"({m.complete_commands.sum()/(m.complete_commands.sum()+m.incomplete_commands.sum())*100:.1f}%)")
    lines.append(f"  Incomplete commands:        "
                 f"{m.incomplete_commands.sum()} "
                 f"({m.incomplete_commands.sum()/(m.complete_commands.sum()+m.incomplete_commands.sum())*100:.1f}%)")

    if subscales:
        lines.append("\n── GODSPEED AVERAGES ──")
        for s in subscales:
            vals = gs[s].dropna()
            lines.append(f"  {s}: mean={vals.mean():.2f}  "
                         f"std={vals.std():.2f}  "
                         f"[{vals.min():.1f}–{vals.max():.1f}]  n={len(vals)}")

    # Correlations
    if subscales:
        gs_for_corr = gs[["user_id"]+subscales].copy()
        merged = m.merge(gs_for_corr, on="user_id", how="inner")
        lines.append(f"\n── SPEARMAN CORRELATIONS (N={len(merged)} matched) ──")
        for p, pl in [("success_rate","Task Success Rate"),
                      ("avg_clarifications","Avg Clarifications")]:
            for s in subscales:
                if merged[p].std()>0 and merged[s].std()>0:
                    r,pv = stats.spearmanr(merged[p], merged[s])
                    sig  = "p<0.05 *" if pv<0.05 else "n.s."
                    lines.append(f"  {pl} vs {s}: r={r:.3f} p={pv:.4f} [{sig}]")

    txt = "\n".join(lines)
    with open(f"{OUT}/user_study_discussion.txt","w") as f:
        f.write(txt)
    print("\nuser_study_discussion.txt saved")
    print(txt)


# MAIN

if __name__ == "__main__":
    print("="*65)
    print("  SHRI User Study: Complete Analysis ")
    print("="*65)

    df  = load_sessions()
    gs  = load_godspeed()
    m   = per_user_metrics(df)

    print("\n[A] Performance...")
    plot_A1_success_rate(m)
    plot_A2_clarification_rate(m)
    plot_A3_outcome_distribution(df)
    plot_A4_combined_metrics(m)

    print("\n[B] Time and steps...")
    plot_B1_episode_duration(df)
    plot_B2_time_metrics(m)
    plot_B3_clarification_steps(df)

    print("\n[C] Commands...")
    plot_C1_command_completeness(m)
    plot_C2_intent_distribution(df)
    plot_C3_most_clarified_commands(df)
    plot_C4_command_heatmap(df)

    print("\n[D] Demographics...")
    plot_D_demographics(df)

    print("\n[E] Correlation...")
    plot_E1_correlation_matrix(m, gs)
    plot_E2_scatter(m, gs)

    print("\n[F] Q-learning...")
    plot_F1_policy_plain_language()
    plot_F2_qvalue_analysis()

    print("\n[G] GodSpeed...")
    plot_G1_godspeed(gs)
    plot_G2_per_user(gs)

    print("\n[Discussion]")
    write_user_study_discussion(m, gs, df)

    files = sorted(os.listdir(OUT))
    print(f"\n{'='*65}")
    print(f"  Done. {len(files)} files saved to: {OUT}/")
    for f in files:
        print(f"  {f}")
    print("="*65)